# 🚀 VideoAI Studio — Kaggle Free GPU Video Factory
### Realistic Fluid Video Generation • 100% FREE (30 Hours/Week) • P100 / Dual T4 (16GB–30GB VRAM)

This Kaggle notebook turns Kaggle's free GPU compute into a cloud video generator for your local **VideoAI Studio** (`http://localhost:5000`).

---
### ⚠️ BEFORE YOU RUN — 2 Required Kaggle Settings (Right Sidebar):
1. **Settings ➔ Accelerator**: Select **`GPU T4 x 2`** or **`GPU P100`**
2. **Settings ➔ Internet**: Toggle **`Internet On`** (requires phone verification on Kaggle account if not done yet)
---

### Step 1: Verify Kaggle GPU Allocation

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("⚠️ NO GPU DETECTED! Go to Notebook Settings on the right and select GPU P100 or T4x2!")

### Step 2: Install Diffusion & Video AI Libraries

In [ ]:
!pip install -q pycloudflared diffusers imageio-ffmpeg

### Step 3: Launch VideoAI Cloud Engine (Temporal Motion Model + Cloudflare Tunnel)
This runs the realistic video generation engine and prints your public tunnel URL.

In [ ]:
# Clean up any previous tunnel/server if re-running cell
!pkill -f cloudflared || true
import os, sys, time, json, threading, uuid, socket
from pathlib import Path
import torch
from fastapi import FastAPI, Request
from fastapi.responses import FileResponse, JSONResponse
import uvicorn
from pycloudflared import try_cloudflare

def get_free_port(start=8000):
    for p in range(start, start + 20):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            if s.connect_ex(('127.0.0.1', p)) != 0:
                return p
    return 8000

PORT = get_free_port(8000)

app = FastAPI(title="VideoAI Studio - Cloud GPU Engine")
OUTPUT_DIR = Path("/kaggle/working/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Global pipeline instance (lazy-loaded on first request)
pipeline = None

def get_pipeline():
    global pipeline
    if pipeline is None:
        print("⚡ Loading Realistic Temporal Video Pipeline onto Kaggle GPU...")
        from diffusers import AnimateDiffPipeline, MotionAdapter, EulerDiscreteScheduler
        adapter = MotionAdapter.from_pretrained("guoyww/animatediff-motion-adapter-v1-5-2", torch_dtype=torch.float16)
        pipeline = AnimateDiffPipeline.from_pretrained(
            "SG161222/Realistic_Vision_V5.1_noVAE",
            motion_adapter=adapter,
            torch_dtype=torch.float16
        ).to("cuda")
        pipeline.scheduler = EulerDiscreteScheduler.from_config(
            pipeline.scheduler.config,
            beta_schedule="linear",
            steps_offset=1,
            clip_sample=False
        )
        pipeline.enable_vae_slicing()
        print("✅ Realistic Video Model successfully initialized on GPU!")
    return pipeline

@app.get("/")
def health_check():
    return {"ok": True, "engine": "Kaggle VideoAI GPU Hub", "status": "online"}

@app.post("/api/generate")
async def generate_video(req: Request):
    data = await req.json()
    prompt = data.get("prompt", "cinematic realistic video, natural motion, 4k, photorealistic")
    negative = data.get("negative_prompt", "blurry, low quality, distorted, static, deformed, oversaturated")
    steps = int(data.get("steps", 25))
    frames = int(data.get("frames", 16))
    
    print(f"🎬 [Kaggle GPU] Generating video for: '{prompt}' (Steps: {steps}, Frames: {frames})...")
    pipe = get_pipeline()
    
    output = pipe(
        prompt=prompt,
        negative_prompt=negative,
        num_inference_steps=steps,
        guidance_scale=7.5,
        num_frames=frames
    )
    
    from diffusers.utils import export_to_video
    vid_id = str(uuid.uuid4())[:8]
    out_file = OUTPUT_DIR / f"video_{vid_id}.mp4"
    export_to_video(output.frames[0], str(out_file), fps=8)
    print(f"✅ [Kaggle GPU] Video generated and saved: {out_file.name}")
    return {"ok": True, "video_url": f"/outputs/{out_file.name}"}

@app.get("/outputs/{filename}")
def serve_output(filename: str):
    p = OUTPUT_DIR / filename
    if p.exists():
        return FileResponse(str(p), media_type="video/mp4")
    return JSONResponse({"error": "File not found"}, status_code=404)

# Start Uvicorn in background thread
server_thread = threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="warning"),
    daemon=True
)
server_thread.start()
time.sleep(3)

# Expose via public Cloudflare Tunnel
tunnel = try_cloudflare(port=PORT)
print("=" * 65)
print("✨ YOUR KAGGLE VIDEO AI CLOUD ENDPOINT URL:")
print(tunnel.tunnel)
print("=" * 65)
print("👉 Copy the URL above and paste it into VideoAI Studio at http://localhost:5000")
print("   under '⚙️ AI Cores & Endpoints' -> 'Google Colab / Kaggle Cloud GPU URL'!")
print("=" * 65)
print("🟢 Kaggle GPU Engine is ACTIVE and ready to render realistic videos...")

# Keep cell running
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("Server stopped.")
